# SPE10: published Darcy and Brinkman spaces

The macrogrid is 6×11 squares, pressure is local Q1, and each face has a continuous P1 trace on 32 segments. The article does not report its local subdivision count; this study varies that count independently. The conforming Q3 comparison uses the article's 768×1408 grid and 9,738,625 unknowns. It is assembled separately in PyMHM with shared basis kernels, not in an independent code.


The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/11815853bdf5761f89f926baa8432e83995f66f4fc532ad3958438637f0ec269/23_spe10_mhm_comparison-companion.zip"
COMPANION_SHA256 = "11815853bdf5761f89f926baa8432e83995f66f4fc532ad3958438637f0ec269"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/23_spe10_mhm_comparison.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Execute the SPE10 physical problem with declared forms

The current controls use the versioned layer-36 tensor on the full
$1200\times2200$ ft reservoir, zero source, pressure one at the bottom,
pressure zero at the top and zero outward Darcy flux on the side walls.
`define_cartesian_darcy` declares the local energy, constant pressure kernel,
physical integral moment, oriented conormal pairings and boundary functional.
The cells below call generic assembly and solve explicitly.

The immediate Darcy control keeps the published 6×11 macrogrid and Q1 local
pressure, with 8×8 local cells and C0 piecewise P1 traces on two segments per
macroface. The article uses 32 trace segments; `--study` executes those original
spaces. The sampled gradient flux and the conservative multiplier are distinct.

In [ ]:
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import assemble, CartesianMacroMesh, FaceSpace, SkeletonSpace
from examples.formulations.darcy import pressure_constraints
from examples.formulations.cartesian_darcy import define_cartesian_darcy, recover_cartesian_darcy
from examples.solve_spe10 import load_layer, pressure_boundary

OUTPUT = ROOT / "examples/results/spe10"
material = load_layer(36)
mesh = CartesianMacroMesh(6, 11, (0, 1200, 0, 2200))
skeleton = SkeletonSpace(
    mesh, tuple(FaceSpace.uniform(1, 2, continuous=True) for _ in mesh.faces)
)
natural = {
    int(face): 0.0 for face in mesh.boundary_faces
    if abs(mesh.normals[face, 0]) > 0.5
}
with threadpool_limits(1):
    definition = define_cartesian_darcy(
        mesh, permeability=material, source=0.0, dirichlet=pressure_boundary,
        neumann=natural, skeleton=skeleton, degree=1, local_refinement=8,
        quadrature_order=3,
    )
    system = assemble(definition.problem)
    solution = recover_cartesian_darcy(
        definition, system,
        system.solve(constraints=pressure_constraints(definition, system)),
    )
assert solution.hybrid.residual < 1e-10
assert np.max(np.abs(solution.conservation_residuals())) < 1e-10
print({"macro_cells": len(mesh.cells), "relative_residual": solution.hybrid.residual,
       "macro_balance_max": float(np.max(np.abs(solution.conservation_residuals())))})

### Independent conforming refinement controls

These Q1 references use the same whole reservoir, tensor pixels, source and
boundary data. Their volume quadrature cuts each cell at the material pixel
interfaces. They are assembled independently of MHM with strong pressure data
and natural side fluxes. Three successively refined fields report their own
physical L2 and H1 increments. These affordable levels do not establish a
converged reference for the channelized tensor; the output states the measured
sensitivity and `--study` supplies the finer published-size acquisition.

In [ ]:
from examples.formulations.conforming import conforming_quadrilateral
from examples.periodic_norms import difference

references = []
with threadpool_limits(1):
    for nx, ny in ((12, 22), (24, 44), (48, 88)):
        reference_mesh = CartesianMacroMesh(nx, ny, (0, 1200, 0, 2200))
        reference_natural = {
            int(face): 0.0 for face in reference_mesh.boundary_faces
            if abs(reference_mesh.normals[face, 0]) > 0.5
        }
        references.append(conforming_quadrilateral(
            reference_mesh, degree=1, permeability=material,
            dirichlet=pressure_boundary, neumann=reference_natural,
            quadrature_order=3,
        ))
    for fine, coarse in zip(references[1:], references[:-1]):
        print(difference(fine, (coarse,)))
print("Refinement sensitivity controls; these levels are not a converged baseline.")

### USFEM Brinkman on the same layer-1 reservoir

This control keeps the original operator: viscosity 0.3, layer-1 isotropic
resistance $0.3/K_x$, zero body force, unit upward inlet velocity, no horizontal
velocity on the side walls, zero vertical side pseudo-traction and zero outlet
pseudo-traction. The local P3/P3 pointwise-2017 GaLS forms and their constant
pressure moment are explicit in `define_flow`. This immediate run uses 32
crisscross macrotriangles, four local triangles per macroelement and one P1
trace segment; `--study` retains the original 264 macrotriangles and finer local
and trace spaces. The diagnostics distinguish macro velocity balance from the
fine-cell divergence norm. Pseudo-traction is the multiplier for the vector
Laplacian formulation, rather than a symmetric-stress traction.

In [ ]:
from examples.formulations.vector import define_flow, flow_constraints, recover_vector
from examples.solve_spe10_brinkman import (
    crisscross_mesh, layer_resistance, inlet_velocity, physical_diagnostics,
)

flow_mesh = crisscross_mesh(2, 4)
flow_skeleton = SkeletonSpace(
    flow_mesh, tuple(FaceSpace.uniform(1) for _ in flow_mesh.faces), 2
)
outlet = {
    int(face): (0.0, 0.0) for face in flow_mesh.boundary_faces
    if flow_mesh.normals[face, 1] > 0.5
}
side = {
    int(face): {1: 0.0} for face in flow_mesh.boundary_faces
    if abs(flow_mesh.normals[face, 0]) > 0.5
}
with threadpool_limits(1):
    flow_definition = define_flow(
        flow_mesh, formulation="usfem", degree=3, local_refinement=2,
        skeleton=flow_skeleton, viscosity=0.3, drag=layer_resistance(),
        dirichlet=inlet_velocity, traction=outlet, traction_components=side,
        stabilization="pointwise-2017", quadrature_order=5,
    )
    flow_system = assemble(flow_definition.problem)
    flow_solution = recover_vector(
        flow_definition, flow_system,
        flow_system.solve(constraints=flow_constraints(flow_definition, flow_system)),
    )
    diagnostics = physical_diagnostics(flow_solution, 2)
assert flow_solution.hybrid.residual < 1e-10
assert diagnostics["macro_divergence_integral_max"] / 1200 < 1e-10
print(diagnostics)

## Checks against the archived published-size runs

The checks below cover file identity, reported spaces, global dimension and conservation. They do not replace the assembly and solve commands in the case documentation. Published curve values are raster samples with a stated uncertainty, not exact field coefficients.


In [ ]:
if HISTORICAL_REPLAY:
    rows = [json.loads(p.read_text()) for p in OUTPUT.glob("darcy-q1-r*-s32.json")]
    aligned = sorted((r for r in rows if r["coefficient_pixels_aligned"]), key=lambda r:r["local_refinement"][0])
    assert len(aligned) >= 5
    for row in aligned:
        assert row["macro_cells"] == 66 and row["degree"] == 1
        assert row["skeleton_degree"] == 1 and row["skeleton_continuous"]
        assert row["free_trace_plus_retained"] == 4257
        assert row["macro_balance_max"] < 1e-11
        assert hashlib.sha256((OUTPUT / row["archive"]).read_bytes()).hexdigest() == row["sha256"]
        print(row["local_refinement"][0], -row["exterior_flux_bottom_right_top_left"][0], row["macro_balance_max"])
    reference = json.loads((OUTPUT / "reference-q3-768x1408-order5.json").read_text())
    assert reference["degrees_of_freedom"] == 9738625
    assert reference["cells"] == 1081344
    assert reference["free_residual_relative"] < 1e-10
    assert hashlib.sha256((OUTPUT/reference["archive"]).read_bytes()).hexdigest() == reference["sha256"]
    comparison = json.loads((OUTPUT / "darcy-comparison.json").read_text())
    print("Raster pressure uncertainty:", comparison["pressure_digitization_uncertainty"])


In [ ]:
if HISTORICAL_REPLAY:
    for name in ("l07-figure-6.png", "l07-figure-7-8.png"):
        display(Image(filename=str(ROOT / "docs/figures/reservoir-papers" / name)))
    for name in ("darcy-fields", "darcy-reference-fields", "darcy-profiles", "darcy-refinement"):
        display(Image(filename=str(ROOT / "docs/figures/spe10" / f"{name}.png")))


The expected flow follows the connected permeable channels and preserves total inflow/outflow. Solid refinement curves use material-aligned cells; dashed curves use cells crossing permeability jumps, with exact intersection quadrature. Exact integration does not add gradient jumps to the Q1 space, so these grid families are not a single monotone convergence sequence. All spatial panels show the actual macrogrid and profiles retain independent one-sided values.

Source: [Paredes, Valentin and Versieux, Section 5.2 and Figures 5–8](https://doi.org/10.1016/j.cam.2023.115415). See the SPE10 case page for the separately specified 2017 Brinkman configuration and its verification.


## Darcy flux and independent classical references

The layer-36 flux figure in Barrenechea et al. (2026) uses triangular P2 local pressure, constant traces and RT2 reconstruction. This differs from the fixed quadrilateral Q1/continuous-P1 MHM computation. The article image is a qualitative field comparison; no vector arrays or numerical flux profile are supplied.

The computed comparisons use the same material and boundary data. Q3 is a separate conforming assembly using PyMHM kernels. MSL_CG is a native conforming P1 assembly; NeoPZ is a native mixed RT0/P0 assembly. Their mesh refinements quantify the uncertainty of each numerical reference. These fields are not exact solutions. Reported L2 differences integrate vector fields on mesh intersections, whereas the maps show identical material-pixel center samples. Every panel includes the 66-square macro partition.

Signed component maps use a blue–white–red palette and a symmetric asinh color scale. Colorbar labels retain physical flux values; MHM and reference use the same scale, while signed differences use their own scale. Zero is white, and no values are clipped or spatially smoothed.


In [ ]:
if HISTORICAL_REPLAY:
    study = json.loads((OUTPUT / "darcy-flux-comparison.json").read_text())
    for record in (study["mhm"], *study["references"].values()):
        archive = OUTPUT / record["archive"]
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == record["sha256"]
        with np.load(archive) as data:
            assert data["flux"].shape == (60, 220, 2)
            assert np.isfinite(data["flux"]).all()
    for method, sequence in study["refinement"].items():
        assert len(sequence) >= 4
        print(method)
        for row in sequence:
            print(row["cells"], row["inlet_flux"], row.get("flux_relative"))
    print("Integrated fixed-MHM comparisons:")
    for row in study["mhm_differences"]:
        print(row["reference"], row["pressure_relative"], row["flux_relative"])
    display(Image(filename=str(ROOT / "docs/figures/reservoir-papers/l09-figure-6.png")))
    for name in ("darcy-flux-q3", "darcy-flux-q3-components", "darcy-flux-classical-references", "darcy-flux-msl", "darcy-flux-neopz", "darcy-flux-reference-refinement"):
        display(Image(filename=str(ROOT / "docs/figures/spe10" / f"{name}.png")))


## Brinkman: the 2017 P3/P3 configuration

The main case uses layer 1, 264 crisscross macrotriangles, 100 fine triangles per macro, local P3/P3 USFEM, and vector DG P1 traces on ten segments per face. Slip walls and zero top pseudotraction follow the article. The minimum-eigenvalue parameter is evaluated pointwise on material pixels; the paper does not specify this spatial convention or its numerical inverse constant. Matching the printed spaces therefore does not establish an identical historical calculation. Its MHM colorbar maxima are 9.090 (velocity magnitude) and 478.9 (pressure); the computed nodal maxima remain different and are printed below.


In [ ]:
if HISTORICAL_REPLAY:
    flow = json.loads((OUTPUT / "flow-layer1-n6x11-p3-r10-s10-q5-pointwise-2017.json").read_text())
    assert flow["macro_cells"] == 264
    assert flow["velocity_degree"] == flow["pressure_degree"] == 3
    assert flow["local_refinement"]**2 == 100
    assert flow["skeleton_degree"] == 1 and flow["skeleton_segments"] == 10
    assert not flow["skeleton_continuous"]
    assert flow["total_skeleton_unknowns"] == 16520
    assert flow["stabilization"] == "pointwise-2017"
    assert hashlib.sha256((OUTPUT / flow["archive"]).read_bytes()).hexdigest() == flow["sha256"]
    check = json.loads((OUTPUT / "flow-quadrature.json").read_text())["rows"][0]
    assert check["velocity"]["relative_to_higher"] < 1e-9
    assert check["pressure"]["relative_to_higher"] < 1e-9
    print("Computed nodal maxima:", flow["velocity_magnitude_nodal_max"], flow["pressure_nodal_max"])
    print("Order-5/order-8 integrated relative differences:", check["velocity"]["relative_to_higher"], check["pressure"]["relative_to_higher"])
    for name in ("brinkman-layer1", "brinkman-layer1-profiles"):
        display(Image(filename=str(ROOT / "docs/figures/spe10" / f"{name}.png")))


The integration study changes quadrature only, leaving the physical finite-element spaces fixed. The expected channel flow and pressure barrier are recovered, but the remaining discrepancy with the published extrema is not hidden or called an exact reproduction. Independent DOLFINx/UFL checks of the three residual-parameter conventions are recorded in `examples/results/usfem-stabilization-operators.json`. They verify matrices and loads, not the historical parameter selection. Source: [Araya et al. (2017), Figures 18–20](https://doi.org/10.1016/j.cma.2017.05.027).


## Classical conforming Taylor–Hood reference

The P2/P1 reference uses native DOLFINx/UFL and PETSc/MUMPS, with no residual stabilization. The layer, operator, units and mixed boundary conditions match the MHM calculation. Pixel-aligned meshes retain exactly the same material. The top traction fixes the pressure constant, so no pressure shift is introduced. This additional classical baseline is distinct from the article's USFEM reference.

Successive-reference differences use exact quadrature on nested fine triangles. MHM/reference L2 differences are integrated over the broken MHM fine triangles, with an independent quadrature-order study. The finest reference remains a numerical approximation; its own refinement change must be read alongside the MHM difference.

In [ ]:
if HISTORICAL_REPLAY:
    th = sorted((json.loads(p.read_text()) for p in OUTPUT.glob("taylor-hood-[0-9]*x[0-9]*.json")), key=lambda r:r["unknowns"])
    assert len(th) >= 4
    for row in th:
        assert row["velocity_degree"] == 2 and row["pressure_degree"] == 1
        assert row["mesh_shape"][0] % 60 == row["mesh_shape"][1] % 220 == 0
        assert row["original_free_residual_relative"] < 1e-10
        assert abs(row["exterior_balance"]) < 1e-6
        assert row["layer_sha256"] == hashlib.sha256((OUTPUT / "layer-1.npz").read_bytes()).hexdigest()
        assert row["sha256"] == hashlib.sha256((OUTPUT / row["archive"]).read_bytes()).hexdigest()
        print(row["mesh_shape"], row["unknowns"], row.get("successive_difference", {}))
    nx, ny = th[-1]["mesh_shape"]
    norms = json.loads((OUTPUT / f"taylor-hood-mhm-{nx}x{ny}.json").read_text())
    assert norms["reference_coefficients_sha256"] == th[-1]["coefficient_sha256"]
    assert norms["mhm_sha256"] == hashlib.sha256((OUTPUT / norms["mhm"]).read_bytes()).hexdigest()
    ordered = sorted(norms["norms"], key=lambda r:r["quadrature_order"])
    for variable in ("velocity", "pressure"):
        key = variable + "_relative"
        final = ordered[-1][key]
        quadrature_change = abs(ordered[-2][key] / final - 1)
        reference_change = th[-1]["successive_difference"][key]
        print(variable, "MHM/reference L2:", final, "quadrature relative change:", quadrature_change,
              "successive-reference L2:", reference_change,
              "reference change / MHM difference:", reference_change/final)
    for name in ("fields", "refinement", "profiles", "profiles-detail"):
        display(Image(filename=str(ROOT / "docs/figures/spe10" / f"brinkman-taylor-hood-{name}.png")))